# A3 — what distinguishes each subgroup

One-vs-rest on PROGENy, CollecTRI TFs, and genes. Comparison matrix is frontend-ready.
TF methylation-silencing reliability is computed separately from completeness.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = False

def cohort_ids():
    import pandas as pd
    assign = V3 / "cluster_assignments.parquet"
    if assign.is_file():
        return pd.read_parquet(assign)["patient_id"].astype(str).str[:12].unique().tolist()
    expr = INTERIM / "intrinsic_expression.parquet"
    if expr.is_file():
        return pd.read_parquet(expr).index.astype(str).str[:12].unique().tolist()
    return None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        ids = cohort_ids()
        if ids is not None:
            kwargs["sample_ids"] = ids
            kwargs.setdefault("n", len(ids))
            kwargs.setdefault("cohort", True)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
import numpy as np
import pandas as pd
from cluster_stats import annotate_clusters, comparison_matrix, mannwhitney_one_vs_rest, per_cluster_significant_pathways, welch_one_vs_rest
from methylation_tf_reliability import methylation_silencing_reliability
from v3_smoke import PATHWAYS, TFS, GENES

preg = json.loads((REF / "preregistered_k.json").read_text())
assign = pd.read_parquet(V3 / "cluster_assignments.parquet") if (V3 / "cluster_assignments.parquet").is_file() else None
path_p = INTERIM / "pathway_activity.parquet"
tf_p = INTERIM / "tf_activity.parquet"
expr_p = INTERIM / "intrinsic_expression.parquet"

used_real = False
if assign is not None and path_p.is_file() and preg.get("k"):
    sub = assign[(assign["method"]=="gmm") & (assign["covariance_type"]=="full") & (assign["k"]==preg["k"])]
    labels = sub.set_index("patient_id")["cluster"]
    pathways = pd.read_parquet(path_p)
    pathways.index = pathways.index.astype(str).str[:12]
    shared = labels.index.intersection(pathways.index)
    if len(shared) > 20:
        used_real = True
        path_prof = mannwhitney_one_vs_rest(pathways.loc[shared], labels.loc[shared].to_numpy(), "pathway")
        if tf_p.is_file():
            tfs = pd.read_parquet(tf_p)
            tfs.index = tfs.index.astype(str).str[:12]
            var = tfs.loc[shared].var().sort_values(ascending=False).head(200).index
            tf_prof = mannwhitney_one_vs_rest(tfs.loc[shared, var], labels.loc[shared].to_numpy(), "tf")
        else:
            tf_prof = pd.DataFrame()
        if expr_p.is_file():
            expr = pd.read_parquet(expr_p)
            expr.index = expr.index.astype(str).str[:12]
            gene_prof = welch_one_vs_rest(expr.loc[shared].iloc[:, :80], labels.loc[shared].to_numpy(), "gene")
        else:
            gene_prof = pd.DataFrame()
        profiles = pd.concat([path_prof, tf_prof, gene_prof], ignore_index=True)

if not used_real:
    print("A3: real pathway/assignment join too thin — not synthesizing.")
    profiles = pd.DataFrame()
    matrix = {}
    annotations = {}
    counts = {}
else:
    matrix = comparison_matrix(profiles)
    counts = per_cluster_significant_pathways(profiles)
    annotations = annotate_clusters(pd.DataFrame(index=shared), labels.loc[shared].to_numpy())

profiles.to_parquet(V3 / "cluster_profiles.parquet")
# markers: top 50 genes per cluster
genes = profiles[profiles["family"]=="gene"].copy()
if not genes.empty:
    markers = genes.sort_values(["cluster", "q"]).groupby("cluster").head(50)
else:
    markers = profiles.sort_values("q").head(50)
markers.to_parquet(V3 / "cluster_markers.parquet")
(V3 / "comparison_matrix.json").write_text(json.dumps(matrix, indent=2))
(V3 / "cluster_annotations.json").write_text(json.dumps(annotations, indent=2))
meth_p = None
for cand in (RAW / "tcga_brca").glob("**/*methylation*") if (RAW / "tcga_brca").exists() else []:
    meth_p = cand
    break
tf_map = {t: [t] for t in (profiles.loc[profiles["family"]=="tf", "feature"].unique() if not profiles.empty else TFS)}
rel = methylation_silencing_reliability(tf_map, None)
rel.to_parquet(V3 / "tf_methylation_reliability.parquet")
print("pathway counts", counts, "reliability source", set(rel["source"]))


In [ ]:
min_sig = min(counts.values()) if counts else 0
gate("NB_A3", "cluster_differential_pathways", int(min_sig), 3, note=f"per-cluster significant pathway counts: {counts}")
